# Data Ingestion
---

## Optical: Sentinel-2
---

In [134]:
import ee
import geemap
import rioxarray as rxr
import pyproj

In [135]:
ee.Authenticate()
ee.Initialize(
    project="multimodal-regression"
)

In [136]:
"""Manually select ROI."""

m = geemap.Map(basemap="SATELLITE")
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [143]:
"""Extract bounding box."""
ee_roi = m.user_roi.bounds()
ee_roi.coordinates().getInfo()

[[[16.758674, 30.263873],
  [16.801201, 30.263873],
  [16.801201, 30.280363],
  [16.758674, 30.280363],
  [16.758674, 30.263873]]]

In [ ]:
"""Get bounding box from raster."""

# Read in raster from file
src_fp = None
img = rxr.open_rasterio(src_fp)

# Get 2D bounds
bounds = img.rio.bounds()
gee_chm_crs_str = f"EPSG:{pyproj.CRS(img.rio.crs).to_2d().to_epsg()}"

# Project bounds to WGS84
transformer = pyproj.Transformer.from_crs(gee_chm_crs_str, "EPSG:4326", always_xy=True)
xmin_wgs, ymin_wgs = transformer.transform(bounds[0], bounds[1])
xmax_wgs, ymax_wgs = transformer.transform(bounds[2], bounds[3])

# Create ee geom obj
ee_roi = ee.Geometry.Rectangle([xmin_wgs, ymin_wgs, xmax_wgs, ymax_wgs])

In [130]:
"""Fetch data."""

# Define processing helper
def process_s2(image):
    """
    Masks clouds and snow  from Sentinel-2 imagery using the
    provided scene classification layer and a probability
    threshold, then returns spectral bands scaled to percent reflectance.
    """
    mask_cld = image.select("MSK_CLDPRB").lt(2)    # pr_cld
    mask_snw = image.select("MSK_SNWPRB").lt(2)    # pr_snw

    spectral_bands = image.select(["B2", "B3", "B4", "B8"])

    return spectral_bands.updateMask(mask_cld).updateMask(mask_snw).divide(10000)


# Filter collection
date_start = "2023-06-01"
date_end = "2023-09-01"
cld_percentage = 5

s2_col = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(ee_roi)
    .filterDate(date_start, date_end)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cld_percentage))
).sort("CLOUDY_PIXEL_PERCENTAGE")

print(f"Images found: {s2_col.size().getInfo()}")

# Process collection
s2_col_processed = s2_col.map(process_s2)             # pixelwise mask and scale
s2_single = s2_col_processed.first().clip(ee_roi)
s2_img_med = s2_col_processed.median().clip(ee_roi)   # median composite (less noisy for structure analysis)

Images found: 9


In [131]:
"""Visualize ee data."""

m = geemap.Map()
true_color_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.3
}

false_color_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0,
    "max": 0.3
}

qa_vis = {
    "bands": ["MSK_CLDPRB"],
}

m.centerObject(ee_roi, zoom=12)
m.addLayer(s2_img_med, true_color_vis, "Median True Color (RGB)")
m.addLayer(s2_img_med, false_color_vis, "Median False Color (NIR,R,G)")

m.addLayer(s2_single, true_color_vis, "Single True Color (RGB)")
m.addLayer(s2_single, false_color_vis, "Single False Color (NIR,R,G)")
m

Map(center=[54.0668137429283, -128.42766899999788], controls=(WidgetControl(options=['position', 'transparent_…

In [126]:
"""Stream data to device."""

# Stream to Xarray
native_projection = s2_col.first().select("B2").projection()
s2_img_med_projected = s2_img_med.setDefaultProjection(native_projection)   # force composite projection to native projection

s2_ds = geemap.ee_to_xarray(
    dataset=s2_img_med_projected,
    geometry=ee_roi,
    scale=10,                      # 10m native Sentinel-2 resolution
    crs="EPSG:32610",              # download in UTM Zone 10N
)

print(s2_ds.info())

xarray.Dataset {
dimensions:
	time = 1 ;
	y = 651 ;
	x = 670 ;

variables:
	float32 B2(time, y, x) ;
		B2:id = B2 ;
		B2:data_type = {'type': 'PixelType', 'precision': 'float', 'min': 0, 'max': 6.553500175476074} ;
		B2:dimensions = [6557, 11175] ;
		B2:origin = [1, 476] ;
		B2:crs = EPSG:32610 ;
		B2:crs_transform = [10, 0, 499980, 0, -10, 6100020] ;
	float32 B3(time, y, x) ;
		B3:id = B3 ;
		B3:data_type = {'type': 'PixelType', 'precision': 'float', 'min': 0, 'max': 6.553500175476074} ;
		B3:dimensions = [6557, 11175] ;
		B3:origin = [1, 476] ;
		B3:crs = EPSG:32610 ;
		B3:crs_transform = [10, 0, 499980, 0, -10, 6100020] ;
	float32 B4(time, y, x) ;
		B4:id = B4 ;
		B4:data_type = {'type': 'PixelType', 'precision': 'float', 'min': 0, 'max': 6.553500175476074} ;
		B4:dimensions = [6557, 11175] ;
		B4:origin = [1, 476] ;
		B4:crs = EPSG:32610 ;
		B4:crs_transform = [10, 0, 499980, 0, -10, 6100020] ;
	float32 B8(time, y, x) ;
		B8:id = B8 ;
		B8:data_type = {'type': 'PixelType', 'precisi

In [ ]:
""""Write data to file."""

# Shape dataset dimensions
s2_ds_2d = s2_ds.squeeze("time", drop=True)

# Write
dst_fp = None
s2_ds_2d.rio.to_raster(
    dst_fp,
    driver="GTiff",
    compress="deflate",   # lossless compression
    tiled=True,           # internal tiles for faster loading in QGIS
)

## SAR: Sentinel-1 C-Band
---